# NetFlow-v3 Feature Pruning, Normalization & Resampling Pipeline
## Production Data Engineering for Cross-Domain IDS & MIA Benchmarking

This notebook implements the end-to-end data processing and resampling pipeline for the **NetFlow-v3 benchmark suite**:
1. **Shortcut Learning Mitigation**: Pruning non-generalizable network identifiers (`IPV4_SRC_ADDR`, `L4_SRC_PORT`, `IPV4_DST_ADDR`, `L4_DST_PORT`, `FLOW_START_MILLISECONDS`, `FLOW_END_MILLISECONDS`).
2. **Exact Vectorized Deduplication**: Eliminating synthetic testbed duplicate flows while strictly preserving legitimate traffic distributions.
3. **Continuous Feature Normalization**: Robust Min-Max scaling of all 47 numerical flow features into $[0, 1]$ with finite handling.
4. **Targeted Resampling Strategy (Per Domain)**:
   - **NF-BoT-IoT-v3 (Smart Home)**: **C-VAE + RUS** (Downsampling dominant DoS/DDoS head classes; C-VAE generative oversampling of rare `Theft`).
   - **NF-ToN-IoT-v3 (Hybrid IIoT / Cloud)**: **C-VAE** (Generative oversampling of ultra-rare minority attacks: `ransomware`, `mitm`, `Backdoor`).
   - **NF-UNSW-NB15-v3 (Campus / Enterprise)**: **None** (Preserves natural distribution).
   - **NF-CSE-CIC-IDS2018-v3 (AWS Cloud Enterprise)**: **None** (Preserves natural distribution).
5. **Standardized Export**: Materializing all processed datasets into `NetFlow_IDS_final/`.


## 1. Environment, DuckDB & CVAE Engine Setup


In [1]:
import os
import sys
import time
from pathlib import Path

# Prevent OpenMP library runtime clash on Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Add current path to import CVAEOversampler
curr_dir = Path.cwd()
if str(curr_dir) not in sys.path:
    sys.path.append(str(curr_dir))

from cvae_oversampling import CVAEOversampler

# Initialize DuckDB with high-throughput configuration
con = duckdb.connect()
con.execute("PRAGMA threads=6;")
con.execute("PRAGMA memory_limit='10GB';")
con.execute("PRAGMA preserve_insertion_order=false;")

print(f"DuckDB Version: {duckdb.__version__}")
print(f"PyTorch Version: {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")


DuckDB Version: 1.5.6
PyTorch Version: 2.13.0+rocm10.0.0 (CUDA Available: True)


## 2. Dataset Discovery & Shortcut Feature Pruning

Features that act as network identifiers or temporal markers allow tree-based and neural network models to exploit **shortcut learning** (memorizing specific IP addresses, destination ports like 445 or 3389, or time windows) rather than learning invariant flow characteristics.

We explicitly drop the 6 non-generalizable columns:
* `IPV4_SRC_ADDR`, `IPV4_DST_ADDR` (Network Layer IP endpoints)
* `L4_SRC_PORT`, `L4_DST_PORT` (Transport Layer port numbers)
* `FLOW_START_MILLISECONDS`, `FLOW_END_MILLISECONDS` (Temporal epoch timestamps)

Remaining schema: **47 standardized continuous/categorical flow features + Label + Attack (49 total columns)**.

We also **pre-deduplicate** all datasets here by registering clean views (`unsw_clean`, `bot_clean`, `ton_clean`, `cic_clean`) that exclude shortcut features and eliminate identical flows upfront across all four benchmark datasets.

In [2]:
# Locate NetFlow datasets dynamically
base_dir = Path.cwd()
possible_roots = [
    base_dir / '../../datasets/NetFlow',
    base_dir / '../datasets/NetFlow',
    base_dir / 'duc_thanh/datasets/NetFlow',
    Path('d:/Swinburne Courses/Project A/duc_thanh/datasets/NetFlow'),
    Path('d:/Swinburne Courses/Project A/datasets/NetFlow')
]

netflow_root = next((p.resolve() for p in possible_roots if p.exists() and p.is_dir()), None)
if not netflow_root:
    raise FileNotFoundError("Could not find datasets/NetFlow directory!")

print(f"NetFlow Source Directory: {netflow_root}")

# Map dataset names to raw CSV files
dataset_files = {
    'unsw': list(netflow_root.glob('NF-UNSW-NB15-v3/*/data/NF-UNSW-NB15-v3.csv'))[0],
    'bot_iot': list(netflow_root.glob('NF-BoT-IoT-v3/*/data/NF-BoT-IoT-v3.csv'))[0],
    'cic_ids': list(netflow_root.glob('NF-CSE-CIC-IDS2018-v3/*/data/NF-CICIDS2018-v3.csv'))[0],
    'ton_iot': list(netflow_root.glob('NF-ToN-IoT-v3/*/data/NF-ToN-IoT-v3.csv'))[0],
}

# Output directory setup
export_dir = netflow_root.parent / 'NetFlow_IDS_final'
export_dir.mkdir(parents=True, exist_ok=True)
print(f"Export Target Directory:  {export_dir.resolve()}")

EXCLUDE_COLS = [
    'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT',
    'FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS'
]
EXCLUDE_SQL = f"EXCLUDE ({', '.join(EXCLUDE_COLS)})"

def get_feature_columns(table_or_view: str) -> list:
    cols = [r[0] for r in con.execute(f"DESCRIBE {table_or_view};").fetchall()]
    return [c for c in cols if c not in ['Label', 'Attack']]

def compute_min_max_expressions(table_name: str, feat_cols: list) -> list:
    agg_exprs = [
        f"MIN(CASE WHEN isfinite({c}::DOUBLE) THEN {c}::DOUBLE ELSE NULL END) AS min_{c}, "
        f"MAX(CASE WHEN isfinite({c}::DOUBLE) THEN {c}::DOUBLE ELSE NULL END) AS max_{c}"
        for c in feat_cols
    ]
    min_max_df = con.execute(f"SELECT {', '.join(agg_exprs)} FROM {table_name};").df()
    scale_exprs = []
    for c in feat_cols:
        c_min = float(min_max_df[f"min_{c}"].iloc[0])
        c_max = float(min_max_df[f"max_{c}"].iloc[0])
        denom = c_max - c_min
        if denom <= 0 or not np.isfinite(denom):
            scale_exprs.append(f"0.0 AS {c}")
        else:
            scale_exprs.append(f'''
                CASE 
                    WHEN NOT isfinite({c}::DOUBLE) THEN 0.0
                    WHEN {c}::DOUBLE <= {c_min} THEN 0.0
                    WHEN {c}::DOUBLE >= {c_max} THEN 1.0
                    ELSE ROUND(({c}::DOUBLE - {c_min}) / {denom}, 6)
                END AS {c}
            '''.strip())
    return scale_exprs

# -------------------------------------------------------------------------
# Pre-deduplicate all datasets (pruning shortcut features & duplicate flows)
# -------------------------------------------------------------------------
clean_views = {
    'unsw': 'unsw_clean',
    'bot_iot': 'bot_clean',
    'ton_iot': 'ton_clean',
    'cic_ids': 'cic_clean',
}

print(">>> Pre-deduplicating all NetFlow datasets...")
for name, view_name in clean_views.items():
    raw_path = str(dataset_files[name]).replace('\\', '/')
    con.execute(f"""
        CREATE OR REPLACE VIEW {view_name} AS
        SELECT DISTINCT * {EXCLUDE_SQL}
        FROM read_csv_auto('{raw_path}');
    """)
    print(f"Registered deduplicated view '{view_name}' for {name}")


NetFlow Source Directory: /run/media/thanh/Ventoy/Project A/duc_thanh/datasets/NetFlow
Export Target Directory:  /run/media/thanh/Ventoy/Project A/duc_thanh/datasets/NetFlow_IDS_final
>>> Pre-deduplicating all NetFlow datasets...
Registered deduplicated view 'unsw_clean' for unsw
Registered deduplicated view 'bot_clean' for bot_iot
Registered deduplicated view 'ton_clean' for ton_iot
Registered deduplicated view 'cic_clean' for cic_ids


## 3. NF-UNSW-NB15-v3 Pipeline
- Attack only: SMOTE-oversampled outperform real
- Get best combination from (use XGBoost for evaluation, **ranked primarily by MCC**):
    - Real Benign + Real Attack
    - Real Benign + OS Attack
    - US Benign + OS Attack
    - US Benign + Real Attack
- For OS Attack use the SMOTE-10k version (`datasets/UNSW-NB15_oversampled/unsw_nb15_attacks_smote_10k.csv`)
- Strict held-out test partition: evaluated on pristine, non-overlapping real flows (zero data leakage).
- Select winning combination based on **MCC (Matthews Correlation Coefficient)**, normalize to $[0, 1]$, and export final dataset.

In [3]:
print(">>> Processing NF-UNSW-NB15-v3 (Benchmarking 4 Resampling Combinations with XGBoost)...")
t0 = time.time()

# Ensure XGBoost and evaluation dependencies are imported
try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess
    print("Installing xgboost in the current environment...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
    from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    roc_auc_score, matthews_corrcoef
)
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------------------
# 1. Locate and Register Oversampled Attack Dataset (SMOTE-10k)
# -------------------------------------------------------------------------
smote_candidates = [
    export_dir.parent / "UNSW-NB15_oversampled" / "unsw_nb15_attacks_smote_10k.csv",
    base_dir / "datasets/UNSW-NB15_oversampled/unsw_nb15_attacks_smote_10k.csv",
    base_dir / "../datasets/UNSW-NB15_oversampled/unsw_nb15_attacks_smote_10k.csv",
    Path("datasets/UNSW-NB15_oversampled/unsw_nb15_attacks_smote_10k.csv")
]
smote_file = next((p.resolve() for p in smote_candidates if p.exists()), None)
if not smote_file:
    raise FileNotFoundError("Could not find datasets/UNSW-NB15_oversampled/unsw_nb15_attacks_smote_10k.csv")

print(f"SMOTE-10k Attack Source: {smote_file}")
feat_cols_unsw = get_feature_columns('unsw_clean')
smote_path_str = str(smote_file).replace('\\', '/')

# -------------------------------------------------------------------------
# 2. Strict Partitioning: Held-Out Test Set (Zero Leakage) & Train Pools
# -------------------------------------------------------------------------
feat_cols_sql = ", ".join(feat_cols_unsw)

# Extract total real benign and attack populations
con.execute(f"""
    CREATE OR REPLACE TABLE unsw_real_benign_all AS
    SELECT {feat_cols_sql}, Label, Attack
    FROM unsw_clean
    WHERE Label = 0;
""")

con.execute(f"""
    CREATE OR REPLACE TABLE unsw_real_attack_all AS
    SELECT {feat_cols_sql}, Label, Attack
    FROM unsw_clean
    WHERE Label = 1;
""")

con.execute(f"""
    CREATE OR REPLACE TABLE unsw_os_attack AS
    SELECT {feat_cols_sql}, Label::BIGINT AS Label, Attack
    FROM read_csv_auto('{smote_path_str}');
""")

# Sample pristine, held-out test partitions from the real distribution
test_benign_cnt = 40000
test_attack_cnt = 10000

con.execute(f"""
    CREATE OR REPLACE TABLE unsw_test_benign AS
    SELECT * FROM unsw_real_benign_all
    USING SAMPLE {test_benign_cnt} (reservoir, 99);

    CREATE OR REPLACE TABLE unsw_test_attack AS
    SELECT * FROM unsw_real_attack_all
    USING SAMPLE {test_attack_cnt} (reservoir, 99);

    CREATE OR REPLACE TABLE unsw_eval_test AS
    SELECT * FROM unsw_test_benign
    UNION ALL
    SELECT * FROM unsw_test_attack;
""")

# Strictly exclude held-out test flows from training pools (EXCEPT ensures zero overlap)
con.execute("""
    CREATE OR REPLACE TABLE unsw_train_benign AS
    SELECT * FROM unsw_real_benign_all
    EXCEPT
    SELECT * FROM unsw_test_benign;

    CREATE OR REPLACE TABLE unsw_train_attack AS
    SELECT * FROM unsw_real_attack_all
    EXCEPT
    SELECT * FROM unsw_test_attack;
""")

n_train_b = con.execute("SELECT COUNT(*) FROM unsw_train_benign;").fetchone()[0]
n_train_a = con.execute("SELECT COUNT(*) FROM unsw_train_attack;").fetchone()[0]
n_os_a = con.execute("SELECT COUNT(*) FROM unsw_os_attack;").fetchone()[0]
n_test = con.execute("SELECT COUNT(*) FROM unsw_eval_test;").fetchone()[0]

print(f"Strict Partitions -> Train Benign: {n_train_b:,} | Train Attack: {n_train_a:,} | OS Attack (SMOTE-10k): {n_os_a:,} | Held-out Test: {n_test:,}")

# Construct Under-Sampled (US) Benign matching attack counts from training benign only
con.execute(f"""
    CREATE OR REPLACE TABLE unsw_us_benign_real AS
    SELECT * FROM unsw_train_benign
    USING SAMPLE {n_train_a} (reservoir, 42);

    CREATE OR REPLACE TABLE unsw_us_benign_os AS
    SELECT * FROM unsw_train_benign
    USING SAMPLE {n_os_a} (reservoir, 42);
""")

# -------------------------------------------------------------------------
# 3. Formulate the 4 Candidate Resampling Combinations
# -------------------------------------------------------------------------
combinations = {
    'Real Benign + Real Attack': 'unsw_comb_real_b_real_a',
    'Real Benign + OS Attack':   'unsw_comb_real_b_os_a',
    'US Benign + OS Attack':     'unsw_comb_us_b_os_a',
    'US Benign + Real Attack':   'unsw_comb_us_b_real_a'
}

con.execute("""
    CREATE OR REPLACE TABLE unsw_comb_real_b_real_a AS
    SELECT * FROM unsw_train_benign UNION ALL SELECT * FROM unsw_train_attack;
""")

con.execute("""
    CREATE OR REPLACE TABLE unsw_comb_real_b_os_a AS
    SELECT * FROM unsw_train_benign UNION ALL SELECT * FROM unsw_os_attack;
""")

con.execute("""
    CREATE OR REPLACE TABLE unsw_comb_us_b_os_a AS
    SELECT * FROM unsw_us_benign_os UNION ALL SELECT * FROM unsw_os_attack;
""")

con.execute("""
    CREATE OR REPLACE TABLE unsw_comb_us_b_real_a AS
    SELECT * FROM unsw_us_benign_real UNION ALL SELECT * FROM unsw_train_attack;
""")

for label, tbl in combinations.items():
    cnt = con.execute(f"SELECT COUNT(*) FROM {tbl};").fetchone()[0]
    print(f"Candidate '{label}': {cnt:,} flows")

# -------------------------------------------------------------------------
# 4. Rigorous XGBoost Evaluation on Pristine Held-Out Real Test Partition
# -------------------------------------------------------------------------
def sanitize_features(df_in, cols):
    """Sanitizes features for XGBoost: converts to numeric, replaces inf/nan, clips extreme values."""
    vals = df_in[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float64)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    vals = np.clip(vals, -1e30, 1e30)
    return vals.astype(np.float32)

df_test = con.execute("SELECT * FROM unsw_eval_test;").df()
X_test = sanitize_features(df_test, feat_cols_unsw)
y_test = df_test['Label'].values

eval_results = []
trained_models = {}

print("\n=== Training & Benchmarking XGBoost Classifiers across 4 Combinations ===")
for comb_name, tbl_name in combinations.items():
    t_train = time.time()
    train_cnt = con.execute(f"SELECT COUNT(*) FROM {tbl_name};").fetchone()[0]
    if train_cnt > 300000:
        df_train_sub = con.execute(f"SELECT * FROM {tbl_name} USING SAMPLE 250000 (reservoir, 42);").df()
    else:
        df_train_sub = con.execute(f"SELECT * FROM {tbl_name};").df()
    
    X_train = sanitize_features(df_train_sub, feat_cols_unsw)
    y_train = df_train_sub['Label'].values
    
    xgb_clf = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        n_jobs=-1,
        random_state=42,
        eval_metric='logloss'
    )
    xgb_clf.fit(X_train, y_train)
    y_pred = xgb_clf.predict(X_test)
    y_prob = xgb_clf.predict_proba(X_test)[:, 1] if hasattr(xgb_clf, 'predict_proba') else y_pred
    
    mcc = matthews_corrcoef(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
    macro_f1 = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)[2]
    roc_auc = roc_auc_score(y_test, y_prob)
    elapsed = time.time() - t_train
    
    eval_results.append({
        'Combination': comb_name,
        'Table': tbl_name,
        'MCC': round(mcc, 4),
        'Accuracy (%)': round(acc * 100, 2),
        'Balanced Acc (%)': round(bal_acc * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)': round(rec * 100, 2),
        'Binary F1': round(f1, 4),
        'Macro F1': round(macro_f1, 4),
        'ROC-AUC': round(roc_auc, 4),
        'Eval Time (s)': round(elapsed, 2)
    })
    trained_models[comb_name] = xgb_clf

# Rank combinations primarily by MCC, then Macro F1
df_eval_summary = pd.DataFrame(eval_results).sort_values(by=['MCC', 'Macro F1', 'Binary F1'], ascending=False).reset_index(drop=True)
print("\n=== XGBoost Benchmark Performance Summary (Ranked by MCC) ===")
display(df_eval_summary)

# -------------------------------------------------------------------------
# 5. Select Winning Combination via MCC, Normalize & Export Final NF-UNSW-NB15-v3
# -------------------------------------------------------------------------
best_record = df_eval_summary.iloc[0]
best_comb_name = best_record['Combination']
best_tbl_name = best_record['Table']
best_mcc = best_record['MCC']
best_macro_f1 = best_record['Macro F1']
print(f"\n>>> Best Performing Resampling Combination (Selected by MCC): '{best_comb_name}' (MCC: {best_mcc}, Macro F1: {best_macro_f1})")

scale_exprs_unsw = compute_min_max_expressions(best_tbl_name, feat_cols_unsw)

# Materialize normalized table
con.execute(f"""
    CREATE OR REPLACE TABLE unsw_final AS
    SELECT 
        {', '.join(scale_exprs_unsw)},
        Label,
        Attack
    FROM {best_tbl_name};
""")

# Export to NetFlow_IDS_final
out_unsw = export_dir / 'NF-UNSW-NB15-v3.csv'
export_path_str = str(out_unsw).replace('\\', '/')
con.execute(f"COPY unsw_final TO '{export_path_str}' (HEADER, DELIMITER ',');")

n_unsw = con.execute("SELECT COUNT(*) FROM unsw_final").fetchone()[0]
print(f"NF-UNSW-NB15-v3 Export Complete: {n_unsw:,} flows | {out_unsw.stat().st_size / (1024**2):.2f} MB | Total Phase Time: {time.time() - t0:.2f}s")
display(con.execute("SELECT Attack, COUNT(*) as cnt FROM unsw_final GROUP BY Attack ORDER BY cnt DESC;").df())


>>> Processing NF-UNSW-NB15-v3 (Benchmarking 4 Resampling Combinations with XGBoost)...


SMOTE-10k Attack Source: /run/media/thanh/Ventoy/Project A/duc_thanh/datasets/UNSW-NB15_oversampled/unsw_nb15_attacks_smote_10k.csv
Strict Partitions -> Train Benign: 1,093,244 | Train Attack: 89,307 | OS Attack (SMOTE-10k): 123,977 | Held-out Test: 50,000
Candidate 'Real Benign + Real Attack': 1,182,551 flows
Candidate 'Real Benign + OS Attack': 1,217,221 flows
Candidate 'US Benign + OS Attack': 247,954 flows
Candidate 'US Benign + Real Attack': 178,614 flows

=== Training & Benchmarking XGBoost Classifiers across 4 Combinations ===

=== XGBoost Benchmark Performance Summary (Ranked by MCC) ===


,Combination,Table,MCC,Accuracy (%),Balanced Acc (%),Precision (%),Recall (%),Binary F1,Macro F1,ROC-AUC,Eval Time (s)
0,Real Benign + Real Attack,unsw_comb_real_b_real_a,0.9997,99.99,99.98,100.00,99.95,0.9997,0.9998,1.0,2.01
1,US Benign + Real Attack,unsw_comb_us_b_real_a,0.9995,99.98,99.99,99.92,100.00,0.9996,0.9998,1.0,0.69
2,US Benign + OS Attack,unsw_comb_us_b_os_a,0.9990,99.97,99.93,99.98,99.86,0.9992,0.9995,1.0,0.93
3,Real Benign + OS Attack,unsw_comb_real_b_os_a,0.9987,99.96,99.91,99.97,99.83,0.9990,0.9994,1.0,1.94



>>> Best Performing Resampling Combination (Selected by MCC): 'Real Benign + Real Attack' (MCC: 0.9997, Macro F1: 0.9998)
NF-UNSW-NB15-v3 Export Complete: 1,182,551 flows | 408.21 MB | Total Phase Time: 13.22s


,Attack,cnt
0,Benign,1093244
1,Exploits,36351
2,Fuzzers,24298
3,Reconnaissance,10757
4,Generic,6810
5,DoS,4832
6,Backdoor,3525
7,Shellcode,1519
8,Analysis,1094
9,Worms,121


## 4. NF-BoT-IoT-v3 Pipeline (Resampling: C-VAE + RUS)

BoT-IoT presents extreme class imbalance:
* `DoS` and `DDoS` constitute over **95%** of all attack flows.
* Rare classes like `Theft` contain only **448** unique flows.

**Strategy**:
1. **Strict Partitioning**: Pristine held-out real test set (`bot_eval_test`) with zero data leakage.
2. **Random Under-Sampling (RUS)**: Downsample majority classes (`DoS`, `DDoS`, `Reconnaissance`) to 25,000 flows each.
3. **Conditional VAE (C-VAE)**: Synthesize high-fidelity flows for `Theft` up to 20,000 samples.
4. **XGBoost Evaluation**: Benchmark detection performance between the **Original (Natural Distribution)** and **Resampled (C-VAE + RUS)** versions using **MCC** as the primary criterion.
5. **Normalization & Export**: Min-Max scale all 47 features to $[0, 1]$ and export to `NF-BoT-IoT-v3.csv`.

In [4]:
print(">>> Processing NF-BoT-IoT-v3 (Resampling: C-VAE + RUS & XGBoost Benchmarking)...")
t0 = time.time()

# Ensure dependencies are available
try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess
    print("Installing xgboost in the current environment...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
    from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    roc_auc_score, matthews_corrcoef
)

def sanitize_features(df_in, cols):
    """Sanitizes features for XGBoost: converts to numeric, replaces inf/nan, clips extreme values."""
    vals = df_in[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float64)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    vals = np.clip(vals, -1e30, 1e30)
    return vals.astype(np.float32)

feat_cols_bot = get_feature_columns('bot_clean')

# -------------------------------------------------------------------------
# 1. Strict Partitioning: Held-Out Test Set (Zero Leakage) & Train Base
# -------------------------------------------------------------------------
# Sample representative held-out test partition across classes (including rare Theft)
con.execute("""
    CREATE OR REPLACE TABLE bot_eval_test AS
    (SELECT * FROM bot_clean WHERE Attack = 'Theft' USING SAMPLE 90 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_clean WHERE Attack = 'Benign' USING SAMPLE 3000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_clean WHERE Attack = 'Reconnaissance' USING SAMPLE 3000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_clean WHERE Attack = 'DoS' USING SAMPLE 5000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_clean WHERE Attack = 'DDoS' USING SAMPLE 5000 (reservoir, 42));

    CREATE OR REPLACE TABLE bot_train_clean AS
    SELECT * FROM bot_clean EXCEPT SELECT * FROM bot_eval_test;
""")

n_test_bot = con.execute("SELECT COUNT(*) FROM bot_eval_test;").fetchone()[0]
n_train_bot_base = con.execute("SELECT COUNT(*) FROM bot_train_clean;").fetchone()[0]
print(f"Strict Partitions -> Train Base: {n_train_bot_base:,} | Held-out Test: {n_test_bot:,}")

# -------------------------------------------------------------------------
# 2. Random Under-Sampling (RUS) on Training Partition
# -------------------------------------------------------------------------
con.execute("""
    CREATE OR REPLACE TABLE bot_rus AS
    (SELECT * FROM bot_train_clean WHERE Attack = 'DoS' USING SAMPLE 25000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_train_clean WHERE Attack = 'DDoS' USING SAMPLE 25000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_train_clean WHERE Attack = 'Reconnaissance' USING SAMPLE 25000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_train_clean WHERE Attack = 'Benign')
    UNION ALL
    (SELECT * FROM bot_train_clean WHERE Attack = 'Theft');
""")

# -------------------------------------------------------------------------
# 3. C-VAE Generative Oversampling for 'Theft'
# -------------------------------------------------------------------------
df_bot_attacks = con.execute("SELECT * FROM bot_rus WHERE Label = 1;").df()
cvae_bot = CVAEOversampler(latent_dim=16, epochs=15, batch_size=128, beta=0.5, random_state=42)
cvae_bot.fit(df_bot_attacks, target_col='Attack', feature_cols=feat_cols_bot, verbose=False)

current_theft = (df_bot_attacks['Attack'] == 'Theft').sum()
needed_theft = 20000 - current_theft
df_syn_theft = cvae_bot.generate_samples('Theft', needed_theft)
df_syn_theft['Label'] = 1
df_syn_theft['Attack'] = 'Theft'
df_syn_theft = df_syn_theft[feat_cols_bot + ['Label', 'Attack']]

con.register('bot_syn_view', df_syn_theft)
con.execute("""
    CREATE OR REPLACE TABLE bot_resampled AS
    SELECT * FROM bot_rus
    UNION ALL BY NAME
    SELECT * FROM bot_syn_view;
""")

# -------------------------------------------------------------------------
# 4. XGBoost Performance Benchmarking (Ranked by MCC)
# -------------------------------------------------------------------------
df_test = con.execute("SELECT * FROM bot_eval_test;").df()
X_test = sanitize_features(df_test, feat_cols_bot)
y_test = df_test['Label'].values
theft_test_mask = (df_test['Attack'] == 'Theft').values

variants = {
    'Original (Natural Distribution)': 'bot_train_clean',
    'Resampled (C-VAE + RUS)':         'bot_resampled'
}

eval_bot_results = []
trained_bot_models = {}

print("\n=== Training & Benchmarking XGBoost Classifiers (Original vs Resampled) ===")
for var_name, tbl_name in variants.items():
    t_v = time.time()
    train_cnt = con.execute(f"SELECT COUNT(*) FROM {tbl_name};").fetchone()[0]
    if train_cnt > 200000:
        df_train_sub = con.execute(f"SELECT * FROM {tbl_name} USING SAMPLE 150000 (reservoir, 42);").df()
    else:
        df_train_sub = con.execute(f"SELECT * FROM {tbl_name};").df()
    
    X_train = sanitize_features(df_train_sub, feat_cols_bot)
    y_train = df_train_sub['Label'].values
    
    xgb_clf = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        n_jobs=-1,
        random_state=42,
        eval_metric='logloss'
    )
    xgb_clf.fit(X_train, y_train)
    y_pred = xgb_clf.predict(X_test)
    y_prob = xgb_clf.predict_proba(X_test)[:, 1] if hasattr(xgb_clf, 'predict_proba') else y_pred
    
    mcc = matthews_corrcoef(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
    macro_f1 = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)[2]
    roc_auc = roc_auc_score(y_test, y_prob)
    theft_rec = (y_pred[theft_test_mask] == 1).mean() * 100 if theft_test_mask.sum() > 0 else 0.0
    elapsed = time.time() - t_v
    
    eval_bot_results.append({
        'Variant': var_name,
        'Table': tbl_name,
        'Train Volume': f"{train_cnt:,}",
        'MCC': round(mcc, 4),
        'Accuracy (%)': round(acc * 100, 2),
        'Balanced Acc (%)': round(bal_acc * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)': round(rec * 100, 2),
        'Binary F1': round(f1, 4),
        'Macro F1': round(macro_f1, 4),
        'ROC-AUC': round(roc_auc, 4),
        'Theft Recall (%)': round(theft_rec, 2),
        'Eval Time (s)': round(elapsed, 2)
    })
    trained_bot_models[var_name] = xgb_clf

df_bot_summary = pd.DataFrame(eval_bot_results).sort_values(by=['MCC', 'Macro F1'], ascending=False).reset_index(drop=True)
print("\n=== NF-BoT-IoT-v3 XGBoost Performance Comparison (Ranked by MCC) ===")
display(df_bot_summary)

# -------------------------------------------------------------------------
# 5. Normalization & Final Export
# -------------------------------------------------------------------------
scale_exprs_bot = compute_min_max_expressions('bot_resampled', feat_cols_bot)
con.execute(f"""
    CREATE OR REPLACE TABLE bot_final AS
    SELECT 
        {', '.join(scale_exprs_bot)},
        Label,
        Attack
    FROM bot_resampled;
""")

out_bot = export_dir / 'NF-BoT-IoT-v3.csv'
export_path_bot = str(out_bot).replace('\\', '/')
con.execute(f"COPY bot_final TO '{export_path_bot}' (HEADER, DELIMITER ',');")

n_bot = con.execute("SELECT COUNT(*) FROM bot_final").fetchone()[0]
print(f"NF-BoT-IoT-v3 Export Complete: {n_bot:,} flows | {out_bot.stat().st_size / (1024**2):.2f} MB | Total Phase Time: {time.time() - t0:.2f}s")
display(con.execute("SELECT Attack, COUNT(*) as cnt FROM bot_final GROUP BY Attack ORDER BY cnt DESC;").df())


>>> Processing NF-BoT-IoT-v3 (Resampling: C-VAE + RUS & XGBoost Benchmarking)...
Strict Partitions -> Train Base: 670,247 | Held-out Test: 4,893

=== Training & Benchmarking XGBoost Classifiers (Original vs Resampled) ===

=== NF-BoT-IoT-v3 XGBoost Performance Comparison (Ranked by MCC) ===


,Variant,Table,Train Volume,MCC,Accuracy (%),Balanced Acc (%),Precision (%),Recall (%),Binary F1,Macro F1,ROC-AUC,Theft Recall (%),Eval Time (s)
0,Original (Natural Distribution),bot_train_clean,"670,247",0.9683,99.92,96.92,99.92,100.00,0.9996,0.9839,0.9998,0.0,3.50
1,Resampled (C-VAE + RUS),bot_resampled,"59,786",0.9300,99.80,99.90,100.00,99.79,0.9990,0.9638,1.0000,0.0,1.96


NF-BoT-IoT-v3 Export Complete: 59,786 flows | 19.28 MB | Total Phase Time: 47.00s


,Attack,cnt
0,Theft,20000
1,Benign,15356
2,DoS,13078
3,DDoS,10278
4,Reconnaissance,1074


## 5. NF-ToN-IoT-v3 Pipeline (Resampling: C-VAE & XGBoost Benchmarking)

ToN-IoT contains minority attack classes (`ransomware`, `mitm`, `Backdoor`) that are heavily underrepresented.

**Strategy**:
1. **Strict Partitioning**: Pristine held-out real test set (`ton_eval_test`) with zero data leakage.
2. **Conditional VAE (C-VAE)**: Synthesize high-fidelity flows for minority classes (`ransomware`, `mitm`, `Backdoor`) up to 25,000 samples each.
3. **XGBoost Evaluation**: Benchmark performance (Original vs Resampled) using **Matthews Correlation Coefficient (MCC)** as the primary evaluation criteria.
4. **Normalization & Export**: Min-Max scale all 47 features into $[0, 1]$ and export to `NF-ToN-IoT-v3.csv`.

In [5]:
print(">>> Processing NF-ToN-IoT-v3 (Resampling: C-VAE & XGBoost Benchmarking)...")
t0 = time.time()

# Ensure dependencies are available
try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess
    print("Installing xgboost in the current environment...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
    from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    roc_auc_score, matthews_corrcoef
)

def sanitize_features(df_in, cols):
    """Sanitizes features for XGBoost: converts to numeric, replaces inf/nan, clips extreme values."""
    vals = df_in[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float64)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    vals = np.clip(vals, -1e30, 1e30)
    return vals.astype(np.float32)

feat_cols_ton = get_feature_columns('ton_clean')

# -------------------------------------------------------------------------
# 1. Strict Partitioning: Held-Out Test Set (Zero Leakage) & Train Base
# -------------------------------------------------------------------------
# Sample representative held-out test partition across classes (including rare ransomware, mitm, Backdoor)
con.execute("""
    CREATE OR REPLACE TABLE ton_eval_test AS
    (SELECT * FROM ton_clean WHERE Attack = 'ransomware' USING SAMPLE 500 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack = 'mitm' USING SAMPLE 500 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack = 'Backdoor' USING SAMPLE 1000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack = 'Benign' USING SAMPLE 10000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack = 'xss' USING SAMPLE 2000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack = 'password' USING SAMPLE 2000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack = 'ddos' USING SAMPLE 2000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack = 'injection' USING SAMPLE 2000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack = 'dos' USING SAMPLE 2000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack = 'scanning' USING SAMPLE 2000 (reservoir, 42));

    CREATE OR REPLACE TABLE ton_train_clean AS
    SELECT * FROM ton_clean EXCEPT SELECT * FROM ton_eval_test;
""")

n_test_ton = con.execute("SELECT COUNT(*) FROM ton_eval_test;").fetchone()[0]
n_train_ton_base = con.execute("SELECT COUNT(*) FROM ton_train_clean;").fetchone()[0]
print(f"Strict Partitions -> Train Base: {n_train_ton_base:,} | Held-out Test: {n_test_ton:,}")

# -------------------------------------------------------------------------
# 2. C-VAE Training on Training Partition
# -------------------------------------------------------------------------
q_ton_train = """
    (SELECT * FROM ton_train_clean WHERE Label = 1 USING SAMPLE 30000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_train_clean WHERE Attack IN ('mitm', 'ransomware', 'Backdoor'))
"""
df_ton_train = con.execute(q_ton_train).df().drop_duplicates()

cvae_ton = CVAEOversampler(latent_dim=24, epochs=15, batch_size=128, beta=0.5, random_state=42)
cvae_ton.fit(df_ton_train, target_col='Attack', feature_cols=feat_cols_ton, verbose=False)

# -------------------------------------------------------------------------
# 3. Synthesize Minority Classes
# -------------------------------------------------------------------------
current_counts = con.execute("""
    SELECT Attack, COUNT(*) as cnt 
    FROM ton_train_clean 
    WHERE Attack IN ('ransomware', 'mitm', 'Backdoor')
    GROUP BY Attack;
""").df().set_index('Attack')['cnt'].to_dict()

rare_targets = {'ransomware': 25000, 'mitm': 25000, 'Backdoor': 25000}
syn_dfs = []
for cls_name, target_cnt in rare_targets.items():
    curr = current_counts.get(cls_name, 0)
    needed = target_cnt - curr
    if needed > 0:
        syn = cvae_ton.generate_samples(cls_name, needed)
        syn['Label'] = 1
        syn['Attack'] = cls_name
        syn = syn[feat_cols_ton + ['Label', 'Attack']]
        syn_dfs.append(syn)

df_ton_syn = pd.concat(syn_dfs, ignore_index=True)
con.register('ton_syn_view', df_ton_syn)

con.execute("""
    CREATE OR REPLACE TABLE ton_resampled AS
    SELECT * FROM ton_train_clean
    UNION ALL BY NAME
    SELECT * FROM ton_syn_view;
""")

# -------------------------------------------------------------------------
# 4. XGBoost Performance Benchmarking (Original vs Resampled via MCC)
# -------------------------------------------------------------------------
df_test = con.execute("SELECT * FROM ton_eval_test;").df()
X_test = sanitize_features(df_test, feat_cols_ton)
y_test = df_test['Label'].values
ransom_mask = (df_test['Attack'] == 'ransomware').values
mitm_mask = (df_test['Attack'] == 'mitm').values

variants = {
    'Original (Natural Distribution)': 'ton_train_clean',
    'Resampled (C-VAE)':               'ton_resampled'
}

eval_ton_results = []
trained_ton_models = {}

print("\n=== Training & Benchmarking XGBoost Classifiers (Original vs Resampled) ===")
for var_name, tbl_name in variants.items():
    t_v = time.time()
    train_cnt = con.execute(f"SELECT COUNT(*) FROM {tbl_name};").fetchone()[0]
    if train_cnt > 250000:
        df_train_sub = con.execute(f"SELECT * FROM {tbl_name} USING SAMPLE 200000 (reservoir, 42);").df()
    else:
        df_train_sub = con.execute(f"SELECT * FROM {tbl_name};").df()
    
    X_train = sanitize_features(df_train_sub, feat_cols_ton)
    y_train = df_train_sub['Label'].values
    
    xgb_clf = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        n_jobs=-1,
        random_state=42,
        eval_metric='logloss'
    )
    xgb_clf.fit(X_train, y_train)
    y_pred = xgb_clf.predict(X_test)
    y_prob = xgb_clf.predict_proba(X_test)[:, 1] if hasattr(xgb_clf, 'predict_proba') else y_pred
    
    mcc = matthews_corrcoef(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
    macro_f1 = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)[2]
    roc_auc = roc_auc_score(y_test, y_prob)
    ransom_rec = (y_pred[ransom_mask] == 1).mean() * 100 if ransom_mask.sum() > 0 else 0.0
    mitm_rec = (y_pred[mitm_mask] == 1).mean() * 100 if mitm_mask.sum() > 0 else 0.0
    elapsed = time.time() - t_v
    
    eval_ton_results.append({
        'Variant': var_name,
        'Table': tbl_name,
        'Train Volume': f"{train_cnt:,}",
        'MCC': round(mcc, 4),
        'Accuracy (%)': round(acc * 100, 2),
        'Balanced Acc (%)': round(bal_acc * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)': round(rec * 100, 2),
        'Binary F1': round(f1, 4),
        'Macro F1': round(macro_f1, 4),
        'ROC-AUC': round(roc_auc, 4),
        'Ransomware Recall (%)': round(ransom_rec, 2),
        'MITM Recall (%)': round(mitm_rec, 2),
        'Eval Time (s)': round(elapsed, 2)
    })
    trained_ton_models[var_name] = xgb_clf

df_ton_summary = pd.DataFrame(eval_ton_results).sort_values(by=['MCC', 'Macro F1'], ascending=False).reset_index(drop=True)
print("\n=== NF-ToN-IoT-v3 XGBoost Performance Comparison (Ranked by MCC) ===")
display(df_ton_summary)

# -------------------------------------------------------------------------
# 5. Normalization & Final Export
# -------------------------------------------------------------------------
scale_exprs_ton = compute_min_max_expressions('ton_resampled', feat_cols_ton)
con.execute(f"""
    CREATE OR REPLACE TABLE ton_final AS
    SELECT 
        {', '.join(scale_exprs_ton)},
        Label,
        Attack
    FROM ton_resampled;
""")

out_ton = export_dir / 'NF-ToN-IoT-v3.csv'
export_path_ton = str(out_ton).replace('\\', '/')
con.execute(f"COPY ton_final TO '{export_path_ton}' (HEADER, DELIMITER ',');")

n_ton = con.execute("SELECT COUNT(*) FROM ton_final").fetchone()[0]
print(f"NF-ToN-IoT-v3 Export Complete: {n_ton:,} flows | {out_ton.stat().st_size / (1024**2):.2f} MB | Total Phase Time: {time.time() - t0:.2f}s")
display(con.execute("SELECT Attack, COUNT(*) as cnt FROM ton_final GROUP BY Attack ORDER BY cnt DESC;").df())


>>> Processing NF-ToN-IoT-v3 (Resampling: C-VAE & XGBoost Benchmarking)...
Strict Partitions -> Train Base: 5,952,732 | Held-out Test: 4,062

=== Training & Benchmarking XGBoost Classifiers (Original vs Resampled) ===

=== NF-ToN-IoT-v3 XGBoost Performance Comparison (Ranked by MCC) ===


,Variant,Table,Train Volume,MCC,Accuracy (%),Balanced Acc (%),Precision (%),Recall (%),Binary F1,Macro F1,ROC-AUC,Ransomware Recall (%),MITM Recall (%),Eval Time (s)
0,Original (Natural Distribution),ton_train_clean,"5,952,732",0.8171,90.77,92.05,81.42,96.83,0.8846,0.9038,0.9730,100.0,0.0,13.35
1,Resampled (C-VAE),ton_resampled,"5,999,895",0.8171,90.74,92.08,81.26,97.04,0.8845,0.9036,0.9726,100.0,0.0,5.64


NF-ToN-IoT-v3 Export Complete: 5,999,895 flows | 1767.65 MB | Total Phase Time: 379.70s


,Attack,cnt
0,xss,2580644
1,Benign,1588228
2,password,697190
3,ddos,538114
4,injection,349951
5,dos,124708
6,scanning,46060
7,ransomware,25000
8,mitm,25000
9,Backdoor,25000


## 6. NF-CSE-CIC-IDS2018-v3 Pipeline (Resampling: None)


In [6]:
print(">>> Processing NF-CSE-CIC-IDS2018-v3 (Resampling: None)...")
t0 = time.time()
feat_cols_cic = get_feature_columns('cic_clean')
scale_exprs_cic = compute_min_max_expressions('cic_clean', feat_cols_cic)

# 1. Materialize normalized table
con.execute(f"""
    CREATE OR REPLACE TABLE cic_final AS
    SELECT 
        {', '.join(scale_exprs_cic)},
        Label,
        Attack
    FROM cic_clean;
""")

# 2. Export to NetFlow_IDS_final
out_cic = export_dir / 'NF-CSE-CIC-IDS2018-v3.csv'
export_path_cic = str(out_cic).replace('\\', '/')
con.execute(f"COPY cic_final TO '{export_path_cic}' (HEADER, DELIMITER ',');")

n_cic = con.execute("SELECT COUNT(*) FROM cic_final").fetchone()[0]
print(f"NF-CSE-CIC-IDS2018-v3 Export Complete: {n_cic:,} flows | {out_cic.stat().st_size / (1024**2):.2f} MB | {time.time() - t0:.2f}s")
con.execute("SELECT Attack, COUNT(*) as cnt FROM cic_final GROUP BY Attack ORDER BY cnt DESC;").df()


>>> Processing NF-CSE-CIC-IDS2018-v3 (Resampling: None)...
NF-CSE-CIC-IDS2018-v3 Export Complete: 14,517,516 flows | 4358.89 MB | 117.70s


,Attack,cnt
0,Benign,13694549
1,DDoS_attacks-LOIC-HTTP,265827
2,DDOS_attack-HOIC,244060
3,DoS_attacks-Hulk,100072
4,SSH-Bruteforce,93980
5,Infilteration,82511
6,DoS_attacks-GoldenEye,27410
7,DoS_attacks-Slowloris,3510
8,Bot,2804
9,DDOS_attack-LOIC-UDP,1723


## 7. Export Catalog Verification & Summary Diagnostics


In [7]:
summary_records = []
for p in sorted(export_dir.glob('*.csv')):
    csv_path_str = str(p).replace('\\', '/')
    t_cnt = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{csv_path_str}')").fetchone()[0]
    size_mb = p.stat().st_size / (1024**2)
    summary_records.append({
        'Dataset': p.stem,
        'Filename': p.name,
        'Flow Count': f"{t_cnt:,}",
        'File Size (MB)': round(size_mb, 2)
    })

df_summary = pd.DataFrame(summary_records)
print("=== Final Exported Datasets in NetFlow_IDS_final ===")
display(df_summary)


=== Final Exported Datasets in NetFlow_IDS_final ===


,Dataset,Filename,Flow Count,File Size (MB)
0,NF-BoT-IoT-v3,NF-BoT-IoT-v3.csv,"59,786",19.28
1,NF-CSE-CIC-IDS2018-v3,NF-CSE-CIC-IDS2018-v3.csv,"14,517,516",4358.89
2,NF-ToN-IoT-v3,NF-ToN-IoT-v3.csv,"5,999,895",1767.65
3,NF-UNSW-NB15-v3,NF-UNSW-NB15-v3.csv,"1,182,551",408.21


## 8. NF-UNSW-NB15-v3 Holdout Partitioning & Stratified 5-Fold Splitting

To ensure reliable, leak-free benchmarking for downstream Intrusion Detection (IDS) and Membership Inference Attack (MIA) models:
1. **Pristine Holdout Partition (15%)**:
   - Reserving an isolated 15% holdout test set from `datasets/final datasets/IDS-binary-classification_MIA/NF-UNSW-NB15-v3.csv` using stratified sampling on `Attack` (preserving both fine-grained attack categories and binary class balance).
   - Saved to `datasets/final datasets/holdout_set/NF-UNSW-NB15-v3.csv`.
2. **Post-Holdout Base Dataset Update**:
   - Updating `datasets/final datasets/IDS-binary-classification_MIA/NF-UNSW-NB15-v3.csv` with the remaining 85% partition ($N = 1,005,168$ flows).
   - Unmentioned datasets (`NF-BoT-IoT-v3.csv`, `NF-CSE-CIC-IDS2018-v3.csv`, `NF-ToN-IoT-v3.csv`) are strictly preserved and untouched.
3. **Stratified 5-Fold Cross-Validation Splitting ($k = 5$)**:
   - Applying `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` across the post-holdout remaining partition.
   - For each fold $k \in \{1, 2, 3, 4, 5\}$, materializing:
     - `datasets/final datasets/train/NF-UNSW-NB15-v3_fold_{k}.csv` (~804,134 flows)
     - `datasets/final datasets/test/NF-UNSW-NB15-v3_fold_{k}.csv` (~201,034 flows)
4. **Verification & Diagnostics**:
   - Verifying partition volume, zero flow leakage, and class distribution stability across all subsets.

In [8]:
# -------------------------------------------------------------------------
# 8. NF-UNSW-NB15-v3 Holdout Partitioning & Stratified 5-Fold Splitting
# -------------------------------------------------------------------------
import os
import time
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

print(">>> Starting NF-UNSW-NB15-v3 Holdout & Stratified 5-Fold Splitting Pipeline...")
t_split_start = time.time()

# 1. Dynamically locate 'datasets/final datasets'
base_dir = Path.cwd()
possible_final_dirs = [
    base_dir / 'datasets/final datasets',
    base_dir / '../datasets/final datasets',
    base_dir / '../../datasets/final datasets',
    base_dir / 'duc_thanh/datasets/final datasets',
    Path('d:/Swinburne Courses/Project A/duc_thanh/datasets/final datasets'),
    Path('d:/Swinburne Courses/Project A/datasets/final datasets'),
]

final_datasets_dir = next((p.resolve() for p in possible_final_dirs if p.exists() and p.is_dir()), None)
if not final_datasets_dir:
    raise FileNotFoundError("Could not locate 'datasets/final datasets' directory!")

print(f"Final Datasets Base Directory: {final_datasets_dir}")

# 2. Directory setup
src_csv = final_datasets_dir / 'IDS-binary-classification_MIA' / 'NF-UNSW-NB15-v3.csv'
holdout_dir = final_datasets_dir / 'holdout_set'
train_dir = final_datasets_dir / 'train'
test_dir = final_datasets_dir / 'test'

holdout_dir.mkdir(parents=True, exist_ok=True)
train_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

holdout_file = holdout_dir / 'NF-UNSW-NB15-v3.csv'

# 3. Load full pristine NF-UNSW-NB15-v3 dataset
# Check active DuckDB session table, or reconstruct if previously partitioned
table_names = [t[0] for t in con.execute("SHOW TABLES;").fetchall()] if 'con' in globals() else []
if 'unsw_final' in table_names:
    print("Loading full dataset from in-memory DuckDB table 'unsw_final'...")
    df_full = con.execute("SELECT * FROM unsw_final;").df()
elif holdout_file.exists():
    print("Holdout partition detected on disk; reconstructing full pristine dataset...")
    df_rem = pd.read_csv(src_csv)
    df_hold = pd.read_csv(holdout_file)
    df_full = pd.concat([df_rem, df_hold], ignore_index=True)
else:
    print(f"Loading full dataset directly from {src_csv}...")
    df_full = pd.read_csv(src_csv)

print(f"Full NF-UNSW-NB15-v3 Shape: {df_full.shape[0]:,} rows x {df_full.shape[1]} features")

# 4. Stratified Holdout Partition (15%)
train_val_df, holdout_df = train_test_split(
    df_full,
    test_size=0.15,
    random_state=42,
    stratify=df_full['Attack']
)

print(f"Holdout Set Partition:       {len(holdout_df):,} rows ({len(holdout_df) / len(df_full) * 100:.2f}%)")
print(f"Post-Holdout Remaining Set:  {len(train_val_df):,} rows ({len(train_val_df) / len(df_full) * 100:.2f}%)")

# Export Holdout Set to datasets/final datasets/holdout_set/
holdout_path_str = str(holdout_file).replace('\\', '/')
con.execute(f"COPY holdout_df TO '{holdout_path_str}' (HEADER, DELIMITER ',');")
print(f"Holdout Set Materialized:    {holdout_file.name} ({holdout_file.stat().st_size / (1024**2):.2f} MB)")

# 5. Overwrite NF-UNSW-NB15-v3.csv in IDS-binary-classification_MIA with post-holdout set
src_path_str = str(src_csv).replace('\\', '/')
con.execute(f"COPY train_val_df TO '{src_path_str}' (HEADER, DELIMITER ',');")
print(f"Updated Post-Holdout Base:   {src_csv.name} ({src_csv.stat().st_size / (1024**2):.2f} MB)")

# 6. Stratified 5-Fold Cross-Validation Splitting (k = 5)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_summary_records = []

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(train_val_df, train_val_df['Attack']), 1):
    fold_train = train_val_df.iloc[train_idx]
    fold_test = train_val_df.iloc[test_idx]
    
    train_file = train_dir / f"NF-UNSW-NB15-v3_fold_{fold_idx}.csv"
    test_file = test_dir / f"NF-UNSW-NB15-v3_fold_{fold_idx}.csv"
    
    train_path_str = str(train_file).replace('\\', '/')
    test_path_str = str(test_file).replace('\\', '/')
    
    con.execute(f"COPY fold_train TO '{train_path_str}' (HEADER, DELIMITER ',');")
    con.execute(f"COPY fold_test TO '{test_path_str}' (HEADER, DELIMITER ',');")
    
    train_sz = train_file.stat().st_size / (1024**2)
    test_sz = test_file.stat().st_size / (1024**2)
    
    fold_summary_records.append({
        'Fold': f"Fold {fold_idx}",
        'Train Filename': train_file.name,
        'Train Flows': f"{len(fold_train):,}",
        'Train Size (MB)': round(train_sz, 2),
        'Test Filename': test_file.name,
        'Test Flows': f"{len(fold_test):,}",
        'Test Size (MB)': round(test_sz, 2),
        'Fold Total': f"{len(fold_train) + len(fold_test):,}"
    })

print(f"\nCompleted Stratified 5-Fold Splitting in {time.time() - t_split_start:.2f}s")
df_folds_summary = pd.DataFrame(fold_summary_records)
display(df_folds_summary)

# 7. Verification of Partition Integrity & Attack Stratification
print("\n=== Verification: Attack Distribution Comparison (Holdout vs Post-Holdout Remaining) ===")
holdout_dist = holdout_df['Attack'].value_counts(normalize=True).rename('Holdout (%)') * 100
rem_dist = train_val_df['Attack'].value_counts(normalize=True).rename('Remaining (%)') * 100
full_dist = df_full['Attack'].value_counts(normalize=True).rename('Full Original (%)') * 100

dist_comparison = pd.concat([full_dist, holdout_dist, rem_dist], axis=1).round(3)
display(dist_comparison)


>>> Starting NF-UNSW-NB15-v3 Holdout & Stratified 5-Fold Splitting Pipeline...
Final Datasets Base Directory: /run/media/thanh/Ventoy/Project A/duc_thanh/datasets/final datasets
Holdout partition detected on disk; reconstructing full pristine dataset...
Full NF-UNSW-NB15-v3 Shape: 1,182,551 rows x 49 features
Holdout Set Partition:       177,383 rows (15.00%)
Post-Holdout Remaining Set:  1,005,168 rows (85.00%)
Holdout Set Materialized:    NF-UNSW-NB15-v3.csv (61.26 MB)
Updated Post-Holdout Base:   NF-UNSW-NB15-v3.csv (346.95 MB)

Completed Stratified 5-Fold Splitting in 24.76s


,Fold,Train Filename,Train Flows,Train Size (MB),Test Filename,Test Flows,Test Size (MB),Fold Total
0,Fold 1,NF-UNSW-NB15-v3_fold_1.csv,"804,134",277.57,NF-UNSW-NB15-v3_fold_1.csv,"201,034",69.38,"1,005,168"
1,Fold 2,NF-UNSW-NB15-v3_fold_2.csv,"804,134",277.56,NF-UNSW-NB15-v3_fold_2.csv,"201,034",69.39,"1,005,168"
2,Fold 3,NF-UNSW-NB15-v3_fold_3.csv,"804,134",277.55,NF-UNSW-NB15-v3_fold_3.csv,"201,034",69.40,"1,005,168"
3,Fold 4,NF-UNSW-NB15-v3_fold_4.csv,"804,135",277.58,NF-UNSW-NB15-v3_fold_4.csv,"201,033",69.38,"1,005,168"
4,Fold 5,NF-UNSW-NB15-v3_fold_5.csv,"804,135",277.54,NF-UNSW-NB15-v3_fold_5.csv,"201,033",69.41,"1,005,168"



=== Verification: Attack Distribution Comparison (Holdout vs Post-Holdout Remaining) ===


,Full Original (%),Holdout (%),Remaining (%)
Attack,,,
Benign,92.448,92.448,92.448
Exploits,3.074,3.074,3.074
Fuzzers,2.055,2.055,2.055
Reconnaissance,0.910,0.909,0.910
Generic,0.576,0.576,0.576
DoS,0.409,0.409,0.409
Backdoor,0.298,0.298,0.298
Shellcode,0.128,0.129,0.128
Analysis,0.093,0.092,0.093
